# Build a Baseline Multi-label Classifier (TF-IDF + One-vs-Rest)

In [1]:
!pip install scikit-learn

**Import**

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report ,accuracy_score, hamming_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from imblearn.over_sampling import RandomOverSampler

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from scipy.sparse import hstack

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

with open("../DepressionEmo/dataset/train.json" ,'r', encoding='utf-8') as f:
    data = [json.loads(line)  for line in f ]
df = pd.DataFrame(data)
df.head()

,id,title,post,text,upvotes,date,emotions,label_id
0,hhcq6e,Found out something awful,My mum had a boyfriend when I was around 6 or ...,Found out something awful ### My mum had a boy...,53,2020-06-28 11:16:59,"[anger, hopelessness, sadness]",10010100
1,d0bobn,I just want to feel wanted ya know?,"Like, I have a ton of friends, I talk to them ...","I just want to feel wanted ya know? ### Like, ...",51,2019-09-06 04:10:27,"[loneliness, sadness, emptiness, hopelessness,...",10111100
2,wy400i,Done,I’m writing this as I sit on the side of the r...,Done ### I’m writing this as I sit on the side...,8,2022-08-26 08:53:35,"[loneliness, hopelessness, sadness, worthlessn...",10111101
3,crkjga,"When nobody else celebrates you, learn to cele...",Feeling unloved can have a huge impact on the ...,"When nobody else celebrates you, learn to cele...",28,2019-08-17 10:28:21,"[loneliness, sadness]",1100
4,zq1lwl,goodbye.,I'm done. I have a bottle of jack danials and ...,goodbye. ### I'm done. I have a bottle of jack...,102,2022-12-19 19:50:52,"[emptiness, hopelessness]",110000


In [16]:
df.columns

Index(['id', 'title', 'post', 'text', 'upvotes', 'date', 'emotions',
       'label_id'],
      dtype='object')

Select relevant features. Let's predict Emotions using selected fatures.

In [17]:
x = df['text']
# x = df[['title','post']]
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['emotions'])  # this turns list of emotions into binary matrix

In [34]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),        # Unigrams + Bigrams
    max_features=10000,        # Keep top 10k tokens
    stop_words='english'       # Optional: remove common English stop words
)
X_vectorized = vectorizer.fit_transform(x)

In [35]:
# 4. Train/Test split
# Example: X = text data, Y = multi-label binary encoded emotions

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [36]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2))),
    ('clf', OneVsRestClassifier(LogisticRegression(solver='liblinear')))
])

pipeline.fit(X_train, y_train)


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
                ('clf',
                 OneVsRestClassifier(estimator=LogisticRegression(solver='liblinear')))])

In [37]:
#Make predictions

y_pred = pipeline.predict(X_test)


In [39]:
# Classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=mlb.classes_))

# Hamming loss
print("Hamming Loss:", hamming_loss(y_test, y_pred))

# Accuracy (not that useful for multi-label but included)
print("Accuracy:", accuracy_score(y_test, y_pred))

Classification Report:

                            precision    recall  f1-score   support

                     anger       0.75      0.59      0.66       339
brain dysfunction (forget)       0.88      0.04      0.08       176
                 emptiness       0.79      0.49      0.60       319
              hopelessness       0.79      0.92      0.85       616
                loneliness       0.86      0.68      0.76       386
                   sadness       0.80      0.99      0.89       669
            suicide intent       0.81      0.31      0.45       227
             worthlessness       0.73      0.68      0.70       424

                 micro avg       0.79      0.70      0.74      3156
                 macro avg       0.80      0.59      0.62      3156
              weighted avg       0.79      0.70      0.71      3156
               samples avg       0.76      0.72      0.70      3156

Hamming Loss: 0.22707100591715976
Accuracy: 0.1455621301775148


C:\Users\cleme\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
